In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

sys.path.append("../src")
from datos import cargar, catalogo, gps_evento
from filtro_adaptado import psd_en, serie_snr, snr_optimo, snr_red, transformada
from plantillas import generador
from senal import irfft

In [ ]:
EVENTOS = ["GW150914", "GW151012", "GW151226", "GW170104", "GW170608", "GW170814", "GW170823"]
DATOS = Path("../datos")
DURACION = 128
DURACION_FFT = 4
DURACION_SEGMENTO = 32
F_MIN = 20
F_MAX = 1024
FS_SALIDA = 2048
INICIO_EVENTO = -24
BORDE_INICIAL = 13
BORDE_FINAL = 0.5
RETRASO_MAXIMO = 0.010
TOLERANCIA = 0.1
SEPARACION_FONDO = 0.2
EVENTOS_INYECCION = ["GW150914", "GW170814"]
INYECCIONES_POR_EVENTO = 5
INICIO_INYECCIONES = -64
MARGEN_INYECCION = 2
FRACCION_RETRASO = 0.8
SNR_INYECTADO = 12
F_MIN_INYECCION = 15
MASA_MINIMA = 6
MASA_MAXIMA = 45
Q_MINIMO = 0.25
GIRO_MINIMO = -0.3
GIRO_MAXIMO = 0.4

In [ ]:
crudos = {}
tiempos = {}
for evento in EVENTOS:
    t0 = gps_evento(evento, DATOS)
    series = {ifo: cargar(ifo, evento, DURACION, DATOS) for ifo in ["H1", "L1"]}
    crudos[evento] = {ifo: serie.value for ifo, serie in series.items()}
    tiempos[evento] = series["H1"].times.value - t0
fs = int(series["H1"].sample_rate.value)
n = DURACION_SEGMENTO * fs
df = fs / n
n_salida = n * FS_SALIDA // fs
indices = np.arange(int(F_MIN / df), int(F_MAX / df) + 1)
banco = np.load(DATOS / "banco.npz")["banco"]
plantilla = generador(indices * df, F_MIN)
print(f"{len(banco)} plantillas")

In [ ]:
def buscar(crudos, tiempos, inicio_segmento):
    inicio = np.searchsorted(tiempos, inicio_segmento)
    t_salida = tiempos[inicio] + np.arange(n_salida) / FS_SALIDA
    validos = (t_salida > inicio_segmento + BORDE_INICIAL) & (t_salida < inicio_segmento + DURACION_SEGMENTO - BORDE_FINAL)
    datos_f = {ifo: transformada(x[inicio : inicio + n], fs, fs // 2) for ifo, x in crudos.items()}
    psd = {ifo: psd_en(np.arange(n // 2 + 1) * df, x, fs, DURACION_FFT) for ifo, x in crudos.items()}
    red_max = np.zeros(validos.sum())
    cual = np.zeros(validos.sum(), dtype=int)
    snr_h1 = np.zeros(validos.sum())
    retraso = np.zeros(validos.sum())
    for a in range(0, len(banco), 50):
        plantillas = np.array([plantilla(*p) for p in banco[a : a + 50]])
        serie_h1 = serie_snr(datos_f["H1"], plantillas, psd["H1"], df, n_salida, indices)
        serie_l1 = serie_snr(datos_f["L1"], plantillas, psd["L1"], df, n_salida, indices)
        red, desplazamiento = snr_red(serie_h1, serie_l1, int(np.ceil(RETRASO_MAXIMO * FS_SALIDA)))
        red = red[:, validos]
        serie_h1 = serie_h1[:, validos]
        desplazamiento = desplazamiento[:, validos]
        mejor = red.argmax(axis=0)
        columnas = np.arange(red.shape[1])
        sube = red[mejor, columnas] > red_max
        red_max[sube] = red[mejor, columnas][sube]
        cual[sube] = a + mejor[sube]
        snr_h1[sube] = serie_h1[mejor, columnas][sube]
        retraso[sube] = desplazamiento[mejor, columnas][sube] / FS_SALIDA
    return t_salida[validos], red_max, cual, snr_h1, retraso

In [ ]:
resultados = {evento: buscar(crudos[evento], tiempos[evento], INICIO_EVENTO) for evento in EVENTOS}

In [ ]:
recuperados = []
for evento in EVENTOS:
    t, red, cual, snr_h1, retraso = resultados[evento]
    i = red.argmax()
    fondo = red[np.abs(t - t[i]) > SEPARACION_FONDO].max()
    snr_l1 = np.sqrt(red[i] ** 2 - snr_h1[i] ** 2)
    parametros = catalogo(evento, DATOS)
    masa_detector = parametros["chirp_mass_source"] * (1 + parametros["redshift"])
    recuperado = abs(t[i]) < TOLERANCIA
    recuperados.append(recuperado)
    print(f"{evento}: SNR de red {red[i]:.1f} (H1 {snr_h1[i]:.1f}, L1 {snr_l1:.1f}; catálogo {parametros['network_matched_filter_snr']}) en t0 {t[i]:+.3f} s, L1 − H1 {1000 * retraso[i]:+.1f} ms, fondo {fondo:.1f}, masa chirp {banco[cual[i], 0]:.1f} M☉ (catálogo ≈{masa_detector:.1f}): {'recuperado' if recuperado else 'no recuperado'}")
print(f"Eventos nuevos recuperados: {sum(recuperados[1:])}/{len(EVENTOS) - 1}")

In [ ]:
fig, ejes = plt.subplots(len(EVENTOS), 1, sharex=True, figsize=(8, 2 * len(EVENTOS)))
for eje, evento in zip(ejes, EVENTOS):
    t, red = resultados[evento][:2]
    eje.plot(t, red, linewidth=0.5)
    eje.axvline(0, color="k", linestyle="--", linewidth=0.8)
    eje.set_ylabel("SNR de red")
    eje.set_title(evento, fontsize=10)
ejes[-1].set_xlabel("Tiempo desde t0 (s)")
fig.tight_layout();

In [ ]:
t, red, cual = resultados["GW150914"][:3]
mejor = banco[cual[red.argmax()]]
forma = np.array([plantilla(*mejor)])
for ifo in ["H1", "L1"]:
    x = crudos["GW150914"][ifo]
    inicio = np.searchsorted(tiempos["GW150914"], INICIO_EVENTO)
    datos_f = transformada(x[inicio : inicio + n], fs, fs // 2)
    t_salida = tiempos["GW150914"][inicio] + np.arange(n_salida) / FS_SALIDA
    cerca = np.abs(t_salida) < TOLERANCIA
    corto = x[np.abs(tiempos["GW150914"]) < 16]
    for nombre, tramo in [("32 s", corto), ("128 s", x)]:
        psd = psd_en(np.arange(n // 2 + 1) * df, tramo, fs, DURACION_FFT)
        print(f"{ifo}, PSD con {nombre}: SNR {serie_snr(datos_f, forma, psd, df, n_salida, indices)[0, cerca].max():.1f}")

In [ ]:
n_total = DURACION * fs
df_total = fs / n_total
indices_total = np.arange(int(F_MIN_INYECCION / df_total), int(F_MAX / df_total) + 1)
en_banda = indices_total * df_total >= F_MIN
forma_inyectada = generador(indices_total * df_total, F_MIN_INYECCION, "IMRPhenomXAS")


def inyectar(x, tiempos, parametros, snr, instante, fase):
    psd = psd_en(np.arange(n_total // 2 + 1) * df_total, x, fs, DURACION_FFT)
    h = forma_inyectada(*parametros)
    escala = snr / snr_optimo(h[en_banda], psd, df_total, indices_total[en_banda])
    espectro = np.zeros(n_total // 2 + 1, dtype=complex)
    espectro[indices_total] = escala * h * np.exp(1j * fase - 2j * np.pi * indices_total * df_total * (instante - tiempos[0]))
    return x + irfft(espectro * fs)

In [ ]:
rng = np.random.default_rng(0)
inyecciones = []
for evento in EVENTOS_INYECCION:
    for _ in range(INYECCIONES_POR_EVENTO):
        q = rng.uniform(Q_MINIMO, 1)
        parametros = (np.exp(rng.uniform(np.log(MASA_MINIMA), np.log(MASA_MAXIMA))), q / (1 + q) ** 2, rng.uniform(GIRO_MINIMO, GIRO_MAXIMO))
        instante = rng.uniform(INICIO_INYECCIONES + BORDE_INICIAL + MARGEN_INYECCION, INICIO_INYECCIONES + DURACION_SEGMENTO - MARGEN_INYECCION)
        retraso = rng.uniform(-FRACCION_RETRASO, FRACCION_RETRASO) * RETRASO_MAXIMO
        inyectados = {
            "H1": inyectar(crudos[evento]["H1"], tiempos[evento], parametros, SNR_INYECTADO / np.sqrt(2), instante, rng.uniform(0, 2 * np.pi)),
            "L1": inyectar(crudos[evento]["L1"], tiempos[evento], parametros, SNR_INYECTADO / np.sqrt(2), instante + retraso, rng.uniform(0, 2 * np.pi)),
        }
        t, red, cual, snr_h1, retraso_medido = buscar(inyectados, tiempos[evento], INICIO_INYECCIONES)
        i = red.argmax()
        inyecciones.append((parametros[0], abs(t[i] - instante) < TOLERANCIA, red[i] / SNR_INYECTADO, banco[cual[i], 0], 1000 * (retraso_medido[i] - retraso)))

In [ ]:
masas_inyectadas, encontradas, cociente, masas_recuperadas, error_retraso = map(np.array, zip(*inyecciones))
print(f"Recuperadas: {encontradas.sum()}/{len(inyecciones)}")
print(f"SNR recuperado / inyectado: mediana {np.median(cociente[encontradas]):.2f}, de {cociente[encontradas].min():.2f} a {cociente[encontradas].max():.2f}")
print(f"Error del retraso L1 − H1: mediana {np.median(np.abs(error_retraso[encontradas])):.1f} ms, máximo {np.abs(error_retraso[encontradas]).max():.1f} ms")
print(f"Masa chirp de la mejor plantilla: error mediano {np.median(np.abs(masas_recuperadas / masas_inyectadas - 1)[encontradas]) * 100:.1f} %")

In [ ]:
plt.scatter(masas_inyectadas[encontradas], cociente[encontradas], label="recuperadas")
plt.scatter(masas_inyectadas[~encontradas], cociente[~encontradas], marker="x", color="r", label="no recuperadas")
plt.axhline(1, color="k", linestyle="--")
plt.xscale("log")
plt.xlabel(r"Masa chirp inyectada (M$_\odot$)")
plt.ylabel("SNR recuperado / inyectado")
plt.legend();